## 向量嵌入

在我們之前的範例中，我們操作的是長度為 `vocab_size` 的高維詞袋向量，而且我們是明確地將低維位置表示向量轉換為稀疏的獨熱表示。這種獨熱表示並不節省記憶體，此外，每個詞語都是獨立處理的，也就是說，獨熱編碼的向量無法表達詞與詞之間的語義相似性。

在本單元中，我們將繼續探索 **News AG** 數據集。首先，讓我們載入數據並從先前的筆記本中獲取一些定義。


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## 甚麼是嵌入（embedding）？

<strong>嵌入</strong>的概念是用較低維度的密集向量來表示詞彙，這些向量某種程度上反映了詞彙的語義。我們稍後會討論如何構建有意義的詞嵌入，但現在先將嵌入視為降低詞向量維度的一種方法。

因此，嵌入層會接受一個詞作為輸入，並產生指定 `embedding_size` 維度的輸出向量。從某種意義上講，它與 `Linear` 層非常相似，但它不是接受獨熱編碼向量，而是能夠接受詞彙編號作為輸入。

透過在網絡的第一層使用嵌入層，我們可以從詞袋模型（bag-of-words）切換到 **embedding bag** 模型，在此模型中，首先將文本中的每個詞轉換成對應的嵌入向量，然後對所有這些嵌入向量計算某些聚合函數，例如 `sum`、`average` 或 `max`。  

![圖像展示用於五詞序列的嵌入分類器。](../../../../../translated_images/zh-MO/embedding-classifier-example.b77f021a7ee67eee.webp)

我們的分類器神經網絡將以嵌入層開始，接著是聚合層，最後在其上方是線性分類器：


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### 處理變量序列大小

因為這種架構，我們需要以某種方式創建給網絡的小批量。在前一單元中，當使用詞袋（BoW）時，小批量中的所有BoW張量大小都相等`vocab_size`，無論我們的文本序列實際長度是多少。一旦我們轉向詞嵌入，我們會得到每個文本樣本中詞數不等的情況，當將這些樣本合並成小批量時，就必須應用一些填充。

這可以通過向數據源提供`collate_fn`函數的相同技術來實現：


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### 訓練嵌入分類器

現在我們已經定義了合適的資料載入器，我們可以使用在上一單元中定義的訓練函數來訓練模型：


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> <strong>注意</strong>：這裡我們只訓練 25k 筆記錄（不到一個完整世代）以節省時間，但你可以繼續訓練，編寫函數以訓練多個世代，並嘗試調整學習率參數以達到更高的準確率。你應該能達到約 90% 的準確率。


### EmbeddingBag 層與可變長度序列表示法

在先前的架構中，我們需要將所有序列補齊到相同長度，才能將它們放入一個小批次中。這並不是表示可變長度序列的最高效方式——另一種做法是使用 **offset** 向量，該向量會保存所有序列在一個大型向量中的偏移量。

![顯示 offset 序列表示法的圖片](../../../../../translated_images/zh-MO/offset-sequence-representation.eb73fcefb29b46ee.webp)

> <strong>注意</strong>：在上圖中，我們顯示了一個字元序列，但在我們的範例中，我們處理的是詞序列。然而，使用 offset 向量來表示序列的一般原理保持不變。

為了使用 offset 表示法，我們使用 [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) 層。它類似於 `Embedding`，但輸入為內容向量和 offset 向量，且它還包含了一個平均層，該層可以是 `mean`、`sum` 或 `max`。

以下是使用 `EmbeddingBag` 的修改網絡：


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

為了準備訓練用的數據集，我們需要提供一個轉換函數來準備偏移向量：


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

注意，與之前所有範例不同，我們的網絡現在接受兩個參數：資料向量和偏移向量，它們的大小不同。同樣地，我們的資料加載器也提供了 3 個值而非 2 個：文字向量和偏移向量都作為特徵提供。因此，我們需要稍微調整訓練函數以處理這個問題：


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## 語義嵌入：Word2Vec

在我們之前的範例中，模型的嵌入層學習將單詞映射到向量表示，然而，此表示並沒有太多語義意義。若能學習出這樣的向量表示，使得相似詞或同義詞在某種向量距離（例如歐氏距離）上對應的向量彼此接近，那將非常理想。

為了做到這點，我們需要用特定方式在大量文本上預先訓練我們的嵌入模型。訓練語義嵌入的最早方法之一稱為 [Word2Vec](https://en.wikipedia.org/wiki/Word2vec)。它基於兩種主要架構，用於產生詞的分佈式表示：

 - <strong>連續詞袋模型</strong>（Continuous bag-of-words，CBoW）— 在此架構中，我們訓練模型從周圍上下文預測一個詞。給定 ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$，模型的目標是從 $(W_{-2},W_{-1},W_1,W_2)$ 預測 $W_0$。
 - <strong>連續跳字模型</strong>（Continuous skip-gram）是 CBoW 的相反。模型使用周圍的上下文字詞視窗來預測當前詞。

CBoW 較快，而 skip-gram 較慢，但對不常見詞的表示效果較佳。

![示意同時包含 CBoW 與 Skip-Gram 算法將單詞轉換為向量。](../../../../../translated_images/zh-MO/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

要使用基於 Google News 資料集預先訓練的 word2vec 嵌入來試驗，我們可以使用 **gensim** 函式庫。以下找出與 'neural' 最相似的詞彙

> **注意：** 當你首次建立詞向量時，下載可能會花一些時間！


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


我們亦可以根據該字計算向量嵌入，以用於訓練分類模型（為清晰起見，我們只顯示向量的首20個分量）： 


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

關於語意嵌入的好處是你可以操作向量編碼來改變語意。例如，我們可以尋找一個單詞，其向量表現會盡可能接近單詞 *king* 和 *woman*，但又盡可能遠離單詞 *man*：


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW 和 Skip-Grams 都是「預測式」的詞向量，因為它們只考慮局部語境。Word2Vec 並未利用全局語境。 

**FastText** 則是在 Word2Vec 基礎上，學習每個字詞及詞內字符 n-gram 的向量表示。這些表示的數值會在每個訓練步驟中平均合併成一個向量。雖然這為預訓練增加了不少額外計算，但這使得詞向量能夠編碼次字詞資訊。 

另一種方法 **GloVe** 利用共現矩陣的概念，使用神經方法將共現矩陣分解為更具表現力且非線性的詞向量。

你可以透過改用 FastText 和 GloVe 的詞向量來玩這個示例，因為 gensim 支援多種不同的詞向量模型。


## 在 PyTorch 中使用預訓練嵌入

我們可以修改上述例子，將我們的嵌入層矩陣預先填入語義嵌入，例如 Word2Vec。我們需要考慮到預訓練嵌入的詞彙表和我們的文本語料庫很可能不匹配，因此我們將為缺失的詞語使用隨機值初始化權重：


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


而家我哋嚟訓練我哋嘅模型。留意訓練個模型所需嘅時間比起之前嘅例子多咗好多，係因為嵌入層嘅大小較大，所以參數數量亦大得多。另外，因為呢個原因，如果我哋想避免過擬合，可能需要用更多嘅例子嚟訓練我哋嘅模型。


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

在我們的案例中，並沒有見到準確度的大幅提升，這很可能是因為詞彙差異很大。
為了解決不同詞彙的問題，我們可以使用以下解決方案之一：
* 在我們的詞彙庫上重新訓練 word2vec 模型
* 用預訓練 word2vec 模型中的詞彙庫加載我們的數據集。加載數據集時可以指定所使用的詞彙庫。

後者的方法似乎更簡單，特別是因為 PyTorch `torchtext` 框架內建對嵌入的支持。例如，我們可以用以下方式實例化基於 GloVe 的詞彙庫：


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


載入的詞彙表具有以下基本操作：
* `vocab.stoi` 字典允許我們將單詞轉換為其字典索引
* `vocab.itos` 則相反 - 將數字轉換為單詞
* `vocab.vectors` 是嵌入向量的陣列，因此要獲取單詞 `s` 的嵌入，我們需要使用 `vocab.vectors[vocab.stoi[s]]`

下面是一個操作嵌入的範例，用以演示方程式 **kind-man+woman = queen**（我必須稍微調整係數才能讓它成立）：


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

為了使用這些嵌入來訓練分類器，我們首先需要使用 GloVe 詞彙編碼我們的數據集：


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

如上所示，所有向量嵌入都存儲在 `vocab.vectors` 矩陣中。這使得通過簡單複製將這些權重加載到嵌入層的權重中變得非常容易：


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

現在讓我們訓練模型，看看是否能得到更好的結果：


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

我們看不到精確度明顯提升的原因之一，是因為我們數據集中的某些詞彙並不存在於預先訓練的 GloVe 詞彙表中，因此它們基本上被忽略了。為了克服這個問題，我們可以在我們的數據集上訓練我們自己的嵌入。 


## 上下文嵌入

傳統預訓練嵌入表示法如 Word2Vec 的一個主要限制是詞義消歧問題。雖然預訓練嵌入可以捕捉詞在上下文中的部分意義，但詞的所有可能含義都編碼在同一個嵌入中。這會在下游模型中造成問題，因為許多詞如「play」一詞，根據使用的上下文，具有不同的意義。

例如詞「play」在下面兩個不同句子中的含義相當不同：
- 我去了劇院看一出<strong>戲劇</strong>。
- 約翰想和他的朋友們<strong>玩耍</strong>。

上述預訓練嵌入將「play」這個詞的兩種含義表示於相同的嵌入向量中。為了解決這一限制，我們需要基於<strong>語言模型</strong>來構建嵌入，該模型在大量語料庫上訓練，<em>瞭解</em>詞如何在不同上下文中組合使用。討論上下文嵌入超出本教學範圍，但我們會在下一單元談到語言模型時回到這個話題。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
